In [13]:
%%writefile RulesForIntents.py

import os
import re
import io
import email
import time
import json
import torch
import shutil
import imaplib
import smtplib
import tempfile
import pandas as pd
import fitz  
from PIL import Image
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from email.header import decode_header
from email.mime.text import MIMEText
from hijri_converter import Hijri
from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.retrievers import BM25Retriever, ParentDocumentRetriever
from langchain.storage import InMemoryStore
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from googleapiclient.discovery import build
from google.oauth2 import service_account
from ArabicOcr import arabicocr  
import gc  
import openpyxl  
import json
import re
from datetime import datetime, timedelta
from hijri_converter import Hijri, Gregorian
from googleapiclient.discovery import build
from google.oauth2 import service_account
import json
import re
import re
import json


from config import tokenizer, llm_pipeline, DEVICE, SERVICE_ACCOUNT_FILE, SCOPES

Overwriting RulesForIntents.py


In [14]:
%%writefile -a RulesForIntents.py

RULES_DIR = "intent_rules"
os.makedirs(RULES_DIR, exist_ok=True)

def load_intent_rules(intent):
    """
    قراءة القواعد الخاصة بنية معينة من ملفها الخاص.
    """
    file_path = os.path.join(RULES_DIR, f"{intent}_rules.txt")
    if os.path.exists(file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                return f.read().strip()
        except Exception as e:
            print(f"⚠ خطأ في قراءة ملف القواعد لـ {intent}: {e}")
            return ""
    return ""

Appending to RulesForIntents.py


In [15]:
%%writefile -a RulesForIntents.py

def deduce_administrative_rule(incoming_body, sent_body, intent):
    """
    وظيفة المحلل: مقارنة الوارد بالصادر لاستنتاج القاعدة الإدارية.
    """
    system_prompt = """
أنت خوارزمية استنتاج إداري. مهمتك تحويل (الإيميل الوارد + الرد الصحيح) إلى "أمر تنفيذي" واحد باللغة العربية.

الشروط الصارمة للمخرجات:
1. اللغة: العربية فقط.
2. الصيغة: سطر واحد فقط بصيغة (الشرط -> التنفيذ).
3. الاختصار: احذف أي كلمات حشو (مثل: يرجى، نود الإفادة، بناء على..).
4. الهيكل المطلوب:
   [الحالة: وصف المرسل والمشكلة] -> [الإجراء: التوجيه الدقيق + المواد النظامية + النبرة]

أمثلة مقبولة:
- [الحالة: شرطة الملز، عطل تكييف] -> [الإجراء: توجيه لشركة الصيانة (الواحة) + ذكر بند الغرامة 3-1 + نبرة حازمة]
- [الحالة: موظف، طلب إجازة] -> [الإجراء: توجيه لمدير الموارد البشرية + بدون مواد قانونية + نبرة رسمية]
"""
    
    user_prompt = f"""
=== الرسالة الواردة (Incoming) ===
{incoming_body}

=== الرد النموذجي الصحيح (Ground Truth) ===
{sent_body}

=== سياق النية ===
{intent}

استخرج القاعدة الذهبية التي تحكم هذا التعامل (ركز على من نوجه له الخطاب وماذا نقتبس من مواد):
"""

    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    
    try:
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        output = llm_pipeline(text_input, temperature=0.1, max_new_tokens=512)[0]['generated_text'].replace(text_input, "").strip()
        return output
    except Exception as e:
        print(f"⚠ خطأ في استنتاج القاعدة: {e}")
        return None

Appending to RulesForIntents.py


In [16]:
%%writefile -a RulesForIntents.py

def update_intent_rules_file(intent, new_rule):
    """
    دمج القاعدة الجديدة مع القواعد القديمة في ملف النية المحدد.
    """
    if not new_rule: return
    
    file_path = os.path.join(RULES_DIR, f"{intent}_rules.txt")
    existing_rules = ""
    
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            existing_rules = f.read()
            
    # استخدام LLM لدمج القواعد وتنظيفها (De-duplication & Merging)
    system_prompt = f"""
أنت مسؤول تدقيق السياسات. لديك قائمة قواعد إدارية (بالعربية) وقاعدة جديدة تريد إضافتها.
مهمتك: دمج القائمة لتكون "مرجعاً سريعاً" (Cheat Sheet) خالياً من التكرار.

التعليمات:
1. إذا كانت القاعدة مكررة، تجاهلها.
2. إذا كانت القاعدة الجديدة أصح أو أدق، استبدل القديمة بها.
3. المخرج النهائي يجب أن يكون قائمة نقاط (Bullet Points) نظيفة.
4. لا تكتب مقدمات ولا خاتمات. فقط القواعد.
"""

    user_prompt = f"""
--- القواعد الحالية ---
{existing_rules}

--- القاعدة الجديدة ---
{new_rule}

أعد صياغة القائمة المحدثة (بالعربية المختصرة):
"""

    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]

    try:
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        updated_rules = llm_pipeline(text_input, temperature=0.1, max_new_tokens=1024)[0]['generated_text'].replace(text_input, "").strip()
        
        # حفظ القواعد المحدثة في الملف
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(updated_rules)
        print(f"✅ تم تحديث ملف القواعد للنية: {intent}")
        
    except Exception as e:
        print(f"⚠ خطأ في تحديث ملف القواعد: {e}")


Appending to RulesForIntents.py
